# Split the Forecast Columns into Train, Validation and Test

## Natural-language request

> Create another notebook which takes all these columns and then splits them properly into train, val, and test.

> Later: these features need to be used: `hotel`, `market_segment`, `distribution_channel`, `agent`, `company`, `is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled`.

"These columns" are the **model inputs recorded in notebook 06**: eight columns, four of which (`is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled`, `company`) are **provisional inputs**, used by team decision although their timing is not verified (notebook 06 and notebook 07, P3a). This notebook takes those inputs, adds the label and the columns the contract needs for scoring, splits every booking into train, validation and test by **arrival date**, checks the split, and saves one file per part.

## Train, val and test are the contract's three periods

The split is not ours to invent. `src/contract.py` fixes the periods (project rules 1 and 4), so the names map one to one:

| This notebook | Contract name | Arrival dates | What it is for |
|---|---|---|---|
| **train** | train | before 2016-07-01 | Fit models, and fit every encoder or scaler. |
| **val** | choose | 2016-07-01 to 2016-12-31 | Compare methods, pick the buffer, enforce the walk ceiling. |
| **test** | proof | from 2017-01-01 | Run **once**, after every choice is frozen (project rule 2). |

## Why not a random split?

- **The forecast is always about future nights.** A random split would train on bookings arriving in 2017 and validate on 2016, so the model would learn from the future. Splitting by arrival date matches how the forecast is used.
- **The contract scores whole nights.** `nightly_table` adds up every booking for a hotel and night. A random split would cut each night's book across train, val and test, and no part would hold a complete night.
- **Proof must stay unseen.** Only a fixed future period can be held back until the end.

## What this notebook does not do

- **It does not read test outcomes.** No cell reads `is_canceled`, `reservation_status` or `reservation_status_date` for test rows. The test label is saved in **a separate file** so a later notebook cannot use it by accident.
- **It does not encode or scale anything.** Encoders must be fit on train only, in the modelling step. Step 5 lists the values they will meet in val and test that never appear in train.
- **It does not make the forecast live** (project rule 7). The split is by final arrival date in the final export. A morning forecast would need timestamped reservation snapshots.

## Setup

Loads the cleaned table from notebook 05 and **the decisions exactly as entered in notebook 06**, by running only notebook 06's `decide-*` cells (the same method as notebook 07). If you change a decision there and rerun this notebook, the split files change with it.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "contract.py").exists():
        repo_root = folder
        break
if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import contract

clean_path = repo_root / "data" / "hotels_clean.pkl"
if not clean_path.exists():
    raise FileNotFoundError("data/hotels_clean.pkl is missing. Run notebooks/05_apply_cleaning_decisions.ipynb first.")
cleaned = pd.read_pickle(clean_path)

outcome_columns = ["is_canceled", "reservation_status", "reservation_status_date"]
to_decide = [column for column in cleaned.columns if column != "is_canceled"]

# Load the decisions from notebook 06 by running only its decision cells.
notebook_06 = json.loads((repo_root / "notebooks" / "06_forecast_column_selection.ipynb").read_text(encoding="utf-8"))
my_decisions = {}
for cell in notebook_06["cells"]:
    if cell.get("id", "").startswith("decide-"):
        exec("".join(cell["source"]))

missing = [column for column in to_decide if column not in my_decisions]
if missing:
    raise ValueError(f"Notebook 06 has no decision for: {missing}")

print(f"Cleaned table: {len(cleaned):,} rows x {cleaned.shape[1]} columns")

Cleaned table: 119,390 rows x 45 columns


## Step 1: which columns go into the split files

Each split file holds three kinds of column, and only the first kind may reach a model:

| Role | Columns | Why it is kept |
|---|---|---|
| **Inputs** | every column notebook 06 marked `input` or `provisional input` | What the model learns from. Provisional inputs are listed separately so every result can say their timing is unverified. |
| **Label** | `is_canceled` | What the model predicts. Never an input (project rule 3). |
| **Scoring only** | `hotel`, `arrival_date`, `adr`, `nights` | `contract.nightly_table` needs them to group bookings into hotel-nights and value an empty room. Using them for scoring does not make them inputs. |

`hotel` is both an input and a grouping key, so it appears once. Everything else (the excluded columns and the outcome columns) is left out of the split files.

In [2]:
inputs = [column for column in to_decide if my_decisions[column]["use"] in ("input", "provisional input")]
provisional = [column for column in inputs if my_decisions[column]["use"] == "provisional input"]
label = "is_canceled"
scoring_only = [column for column in contract.REQUIRED if column not in inputs and column != label]

# Hard guards: no outcome column is an input, and the label is never an input (project rule 3).
assert not set(inputs) & set(outcome_columns), "An outcome column is marked as an input"
assert label not in inputs

kept = inputs + [label] + scoring_only
display(pd.DataFrame({
    "column": kept,
    "role": [my_decisions[column]["use"] for column in inputs] + ["label"] + ["scoring only"] * len(scoring_only),
    "type": [str(cleaned[column].dtype) for column in kept],
}).set_index("column"))
print(f"{len(inputs)} inputs ({len(provisional)} provisional), 1 label, {len(scoring_only)} scoring-only columns; "
      f"{cleaned.shape[1] - len(kept)} columns left out.")

,role,type
column,,
hotel,input,str
market_segment,input,str
distribution_channel,input,str
is_repeated_guest,provisional input,int64
previous_cancellations,provisional input,int64
previous_bookings_not_canceled,provisional input,int64
agent,input,str
company,provisional input,str
is_canceled,label,int64


8 inputs (4 provisional), 1 label, 3 scoring-only columns; 33 columns left out.


## Step 2: split by arrival date

`contract.split_periods` sets the boundaries; nothing here chooses a date. The table shows each part's size, dates and hotel-nights. The cancellation rate is shown for **train and val only**: the test rate is an outcome and stays sealed.

In [3]:
train, val, test = contract.split_periods(cleaned[kept])
splits = {"train": train, "val": val, "test": test}

summary = pd.DataFrame({
    name: {
        "contract name": {"train": "train", "val": "choose", "test": "proof"}[name],
        "bookings": f"{len(part):,}",
        "share of bookings": f"{len(part) / len(cleaned):.1%}",
        "first arrival": f"{part['arrival_date'].min():%Y-%m-%d}",
        "last arrival": f"{part['arrival_date'].max():%Y-%m-%d}",
        "hotel-nights": f"{part.groupby(['hotel', 'arrival_date']).ngroups:,}",
        "City Hotel bookings": f"{(part['hotel'] == 'City Hotel').sum():,}",
        "Resort Hotel bookings": f"{(part['hotel'] == 'Resort Hotel').sum():,}",
        "cancellation rate": "sealed" if name == "test" else f"{part[label].mean():.1%}",
    }
    for name, part in splits.items()
})
display(summary)

,train,val,test
contract name,train,choose,proof
bookings,"49,157","29,546","40,687"
share of bookings,41.2%,24.7%,34.1%
first arrival,2015-07-01,2016-07-01,2017-01-01
last arrival,2016-06-30,2016-12-31,2017-08-31
hotel-nights,732,368,486
City Hotel bookings,"31,623","20,199","27,508"
Resort Hotel bookings,"17,534","9,347","13,179"
cancellation rate,35.8%,36.9%,sealed


## Step 3: check the split

The cell below stops with an error if any of these fails:

1. Every booking lands in **exactly one** part: no booking is lost and none is in two parts.
2. The parts **follow each other in time**: every train arrival is before every val arrival, and every val arrival is before every test arrival.
3. **No hotel-night is cut** across two parts, so every night's book is complete inside one part.
4. Both hotels appear in every part.
5. The inputs have **no missing values** in any part.

In [4]:
# 1. Every booking in exactly one part.
all_index = pd.Index(train.index).append(val.index).append(test.index)
assert all_index.is_unique, "A booking is in more than one part"
assert len(all_index) == len(cleaned) and set(all_index) == set(cleaned.index), "A booking is missing"

# 2. Parts follow each other in time.
assert train["arrival_date"].max() < val["arrival_date"].min()
assert val["arrival_date"].max() < test["arrival_date"].min()

# 3. No hotel-night is cut across parts.
night_keys = {name: set(zip(part["hotel"], part["arrival_date"])) for name, part in splits.items()}
assert not night_keys["train"] & night_keys["val"]
assert not night_keys["val"] & night_keys["test"]
assert not night_keys["train"] & night_keys["test"]

# 4. Both hotels in every part.
for name, part in splits.items():
    assert set(part["hotel"]) == set(cleaned["hotel"]), f"{name} is missing a hotel"

# 5. No missing input values (feature values only; no outcome is read).
missing_inputs = pd.DataFrame({name: part[inputs].isna().sum() for name, part in splits.items()})
assert missing_inputs.values.sum() == 0, missing_inputs[missing_inputs.sum(axis=1) > 0]

print("All five checks pass.")

All five checks pass.


## Step 4: is every training label settled before validation starts?

A time split is only clean if the **label** of a training booking was already known at the boundary. A booking is cancelled **before** it arrives, so a train booking (arrival before 2016-07-01) should have its cancellation recorded before 2016-07-01. If some were recorded later, the model would train on information from the validation period.

The check reads `reservation_status_date` **for cancelled train and val bookings only**. It is used to check the split and never as an input. Test is not read.

In [5]:
boundaries = {"train": contract.TRAIN_END, "val": contract.PROOF_START}
for name, boundary in boundaries.items():
    part = cleaned.loc[splits[name].index]
    cancelled_on = pd.to_datetime(part.loc[part[label] == 1, "reservation_status_date"])
    late = int((cancelled_on >= boundary).sum())
    print(f"{name:>5}: {len(cancelled_on):,} cancelled bookings; latest cancellation recorded "
          f"{cancelled_on.max():%Y-%m-%d}; {late} recorded on or after {boundary}")
    assert late == 0, f"{name} has labels that were only known after {boundary}"

train: 17,585 cancelled bookings; latest cancellation recorded 2016-06-30; 0 recorded on or after 2016-07-01
  val: 10,894 cancelled bookings; latest cancellation recorded 2016-12-31; 0 recorded on or after 2017-01-01


## Step 5: input values that never appear in train

A model or encoder fit on train has never seen these values, so it must handle them (for example, an "unknown" category) instead of failing. This reads **feature values only**, never an outcome. It does not choose anything: it states what the encoder must cope with.

Two points for the modelling step:

- **Fit every encoder, scaler or imputer on train only**, then apply it unchanged to val and test.
- `agent` and `company` are IDs stored as text (`none` means no agent or company). They are labels, never quantities.
- `is_repeated_guest` is 0/1 and the two prior counts are whole numbers. The last table shows how often each is non-zero in each part. Notebook 05 (decision C6) flagged that the share with prior cancellations falls sharply after train, so a model fit on train meets a different mix in val and test.

In [6]:
categorical_inputs = [column for column in inputs if cleaned[column].dtype.kind not in "iufb"]
numeric_inputs = [column for column in inputs if column not in categorical_inputs]

rows = []
for column in categorical_inputs:
    seen = set(train[column])
    row = {"column": column, "values in train": len(seen)}
    for name in ["val", "test"]:
        new = ~splits[name][column].isin(seen)
        row[f"{name}: new values"] = splits[name].loc[new, column].nunique()
        row[f"{name}: rows with a new value"] = f"{new.sum():,} ({new.mean():.1%})"
    rows.append(row)
print("Categorical inputs:")
display(pd.DataFrame(rows).set_index("column"))

print("Numeric inputs (range in each part):")
display(pd.DataFrame({name: part[numeric_inputs].agg(["min", "max"]).T.apply(lambda r: f"{r['min']:g} to {r['max']:g}", axis=1)
                      for name, part in splits.items()}))

print("Share of bookings with a non-zero value:")
display(pd.DataFrame({name: part[numeric_inputs].gt(0).mean().map("{:.1%}".format) for name, part in splits.items()}))

Categorical inputs:


,values in train,val: new values,val: rows with a new value,test: new values,test: rows with a new value
column,,,,,
hotel,2,0,0 (0.0%),0,0 (0.0%)
market_segment,8,0,0 (0.0%),0,0 (0.0%)
distribution_channel,5,0,0 (0.0%),0,0 (0.0%)
agent,219,66,499 (1.7%),69,"1,998 (4.9%)"
company,214,85,332 (1.1%),89,607 (1.5%)


Numeric inputs (range in each part):


,train,val,test
is_repeated_guest,0 to 1,0 to 1,0 to 1
previous_cancellations,0 to 26,0 to 6,0 to 6
previous_bookings_not_canceled,0 to 36,0 to 58,0 to 72


Share of bookings with a non-zero value:


,train,val,test
is_repeated_guest,3.3%,2.7%,3.4%
previous_cancellations,11.8%,1.3%,0.7%
previous_bookings_not_canceled,2.7%,2.9%,3.5%


## Step 6: save the split

Writes to `data/splits/` (inside `data/`, which `.gitignore` keeps out of the repository, like `hotels_clean.pkl`). Every file keeps the **original row number** of the cleaned table as its index, so each booking can be traced back.

Each part is saved twice: as `.pkl` (keeps the column types exactly) and as `.csv` (opens in any tool).

| File (`.pkl` and `.csv`) | Contents |
|---|---|
| `train` | Inputs, label, scoring-only columns |
| `val` | Inputs, label, scoring-only columns |
| `test` | Inputs and scoring-only columns. **No label.** |
| `test_labels` | The test label only. **Open once, at the end** (project rule 2). |
| `split_columns.json` | Which column has which role, the boundaries, and the row counts |

In the CSV files the original row number is the first column, `row`. When reading a CSV back, read `agent` and `company` as text so IDs such as `9` stay labels, not numbers: `pd.read_csv(path, index_col="row", parse_dates=["arrival_date"], dtype={"agent": str, "company": str})`.

To score test at the end: `test.join(test_labels)` gives the columns `contract.nightly_table` needs. The cell reloads every file and checks it before finishing.

In [7]:
split_dir = repo_root / "data" / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

to_save = {"train": train, "val": val, "test": test.drop(columns=[label]), "test_labels": test[[label]]}
for name, frame in to_save.items():
    frame.to_pickle(split_dir / f"{name}.pkl")
    frame.to_csv(split_dir / f"{name}.csv", index_label="row")

manifest = {
    "source": "data/hotels_clean.pkl (notebook 05); inputs from notebook 06 decide-* cells",
    "inputs": inputs,
    "provisional_inputs": provisional,
    "label": label,
    "scoring_only": scoring_only,
    "periods": {
        "train": {"contract_name": "train", "arrivals": f"before {contract.TRAIN_END}", "rows": len(train)},
        "val": {"contract_name": "choose", "arrivals": f"{contract.TRAIN_END} to before {contract.PROOF_START}", "rows": len(val)},
        "test": {"contract_name": "proof", "arrivals": f"from {contract.PROOF_START}", "rows": len(test),
                 "note": "label in test_labels.pkl; open once, after every choice is frozen"},
    },
}
(split_dir / "split_columns.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

# Reload and check every file.
reloaded = {name: pd.read_pickle(split_dir / f"{name}.pkl") for name in ["train", "val", "test", "test_labels"]}
assert reloaded["train"].equals(train) and reloaded["val"].equals(val)
assert list(reloaded["test"].columns) == [c for c in kept if c != label]
assert reloaded["test_labels"].index.equals(test.index) and list(reloaded["test_labels"].columns) == [label]
assert sum(len(reloaded[name]) for name in ["train", "val", "test"]) == len(cleaned)

for name, frame in to_save.items():
    from_csv = pd.read_csv(split_dir / f"{name}.csv", index_col="row", dtype={"agent": str, "company": str},
                           parse_dates=["arrival_date"] if "arrival_date" in frame else False)
    assert from_csv.index.equals(frame.index) and list(from_csv.columns) == list(frame.columns)
    assert from_csv.astype(str).equals(frame.astype(str)), f"{name}.csv does not match {name}.pkl"

for name, frame in reloaded.items():
    print(f"Saved data/splits/{name}.pkl and {name}.csv: {frame.shape[0]:,} rows x {frame.shape[1]} columns")
print("Saved data/splits/split_columns.json")

Saved data/splits/train.pkl and train.csv: 49,157 rows x 12 columns
Saved data/splits/val.pkl and val.csv: 29,546 rows x 12 columns
Saved data/splits/test.pkl and test.csv: 40,687 rows x 11 columns
Saved data/splits/test_labels.pkl and test_labels.csv: 40,687 rows x 1 columns
Saved data/splits/split_columns.json


## Summary

- **The split is the contract's**: train = train, val = choose, test = proof, by arrival date. There is no random split.
- **Every booking is in exactly one part**, the parts follow each other in time, no hotel-night is cut, and both hotels appear in every part.
- **Every train and val label was settled before the next part begins**, so training does not use information from a later period.
- **The test label is in its own file** and its cancellation rate was never computed.
- **Some `agent` and `company` values in val and test never appear in train** (Step 5). Encoders must be fit on train only and must handle unknown values.
- **Four inputs are provisional** (`is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled`, `company`). Every result that uses them must say their timing is unverified; a good val score does not make them clean.

**Next (modelling step):**

1. Load `train.pkl` and `val.pkl`. Fit encoders and models on train; compare every candidate against the flat-rule baseline on val with `contract.nightly_table` and `contract.night_cost`, reporting euros, empty rooms and walked guests (rule 5) and enforcing the walk ceiling (rule 6).
2. Freeze the method, inputs and buffer.
3. Only then join `test_labels.pkl` to `test.pkl` and score test once.